In [1]:
import numpy as np

"""시뮬레이션 결과파일에서 데이터 가져오기 (중차량 비율 5~15%)"""
import sqlite3
import pandas as pd
import win32com.client  # Windows COM 인터페이스 사용
import os
import xml.etree.ElementTree as ET
####################### 입력 값 ############################
# 폴더 경로 설정
folder_path = r"E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\ToyNetwork\2"  # 폴더 경로 입력 받아야함 추후에 함수에 파라미터로 처리 예정
# 웜-업 타임
from_time = 1800
# 진입부, 출입부 링크 번호
link_no = "1" # 파라미터 처리
link_no2 = "3"

# 중차량 반복 값
relflow_start = 20  # 시작(%)
relflow_final = 30 # 마지막(%)
relflow_interval = 5 # 간격(%)

# 종단경사 반복 값
gradient_start = 3  # 시작(%)
gradient_final = 9 # 마지막(%)
gradient_interval = 2 # 간격(%)


# 교통량
volume_list = np.array([
    [1190, 1182, 1172, 1162, 1154, 1144], # 600
    [2678, 2658, 2636, 2616, 2596, 2576], # 1350
    [4364, 4330, 4296, 4264, 4230, 4198]  # 2200
])

# 차량 비율
vehicle1 = np.array([86.24, 81.70, 77.16, 72.62, 68.09, 63.55]) # 소형차
vehicle2 = np.array([8.76, 8.30, 7.84, 7.38, 6.91, 6.45])   # 버스
vehicle3 = np.array([3.56, 7.13, 10.69, 14.26, 17.82, 21.38])   # 소형화물
vehicle4 = np.array([1.27, 2.53, 3.80, 5.07, 6.34, 7.60])   # 중형화물
vehicle5 = np.array([0.17, 0.34, 0.51, 0.68, 0.84, 1.01])   # 대형화물
###########################################################
# 폴더 내 파일 목록 가져오기
file_list = os.listdir(folder_path)
inpx_files = [file for file in file_list if file.endswith(".inpx")]

# VISSIM 실행 및 연결
Vissim = win32com.client.Dispatch("Vissim.Vissim")  # Vissim 인스턴스 생성
simulation_run_count = 0  # VISSIM 실행 횟수 카운트
senario_start = 0

for i in range(len(inpx_files)):
    file_name = inpx_files[i]
    file_path = folder_path + "\\" +file_name
    print("file_path : ", file_path)
    for v in range(len(volume_list)):
        for r in range(relflow_start,relflow_final+1,relflow_interval):
            a = int(r/relflow_interval-1)
            veh = np.array([vehicle1[a], vehicle2[a], vehicle3[a], vehicle4[a], vehicle5[a]]) * 0.01
            volume = volume_list[v][a]

            for g in range(gradient_start,gradient_final+1, gradient_interval):
                # 종단경사 설정
                new_gradient = g * -0.01
                new_gradient2 = g * 0.01

                # XML 파일 파싱
                tree = ET.parse(file_path)
                root = tree.getroot()

                # <link> 태그 찾기
                for link in root.findall(".//link"):
                    if link.get("no") == str(link_no):  # 특정 no 값을 가진 <link> 찾기
                        link.set("gradient", str(new_gradient))  # gradient 속성 변경
                    if link.get("no") == str(link_no2):  # 특정 no 값을 가진 <link> 찾기
                        link.set("gradient", str(new_gradient2))  # gradient 속성 변경

                # <vehicleCompositions> 태그 내 <vehicleComposition name="Default"> 찾기
                for vehicle_comp in root.findall(".//vehicleComposition"):
                    if vehicle_comp.get("name") == "com1":

                        # <vehCompRelFlows> 태그 찾기
                        vehCompRelFlows = vehicle_comp.find("vehCompRelFlows")
                        if vehCompRelFlows is not None:
                            # <vehicleCompositionRelativeFlow> 태그 찾기
                            for relflow in vehCompRelFlows.findall("vehicleCompositionRelativeFlow"):
                                vehType = relflow.get("vehType")
                                if(vehType == "100"): # 승용차
                                    relflow.set("relFlow", str(round(veh[0],4)))
                                if(vehType == "300"): # 버스
                                    relflow.set("relFlow", str(round(veh[1],4)))
                                if(vehType == "630"): # 소형화물
                                    relflow.set("relFlow", str(round(veh[2],4)))
                                if(vehType == "640"): # 중형화물
                                    relflow.set("relFlow", str(round(veh[3],4)))
                                if(vehType == "650"): # 대형화물
                                    relflow.set("relFlow", str(round(veh[4],4)))
                for vehicle_input in root.findall(".//vehicleInput"):
                    if vehicle_input.get("no") == "1":
                        # <timeIntVehVols> 태그 찾기
                        vehicleInput = vehicle_input.find("timeIntVehVols")
                        if vehicleInput is not None:
                            for vol in vehicleInput.findall("timeIntervalVehVolume"):
                                vol.set("volume", str(round(volume)))

                # 변경된 XML을 원본 파일에 덮어쓰기
                tree.write(file_path, encoding="utf-8", xml_declaration=True)

                # VISSIM 네트워크 로드
                Vissim.LoadNet(file_path)
                Vissim.Graphics.CurrentNetworkWindow.SetAttValue('QuickMode', 1) # 퀵 모드 활성화

                sim_volume = 0
                vvs = Vissim.Net.VehicleInputs.GetAll()
                    # ▶ 교통량 설정
                for vv in vvs:
                    name = vv.AttValue("Name")
                    if name in "test":
                        sim_volume = vv.AttValue("Volume(1)")

                Veh_composition_number = 1 # Vehicle Composition 번호
                Rel_Flows = Vissim.Net.VehicleCompositions.ItemByKey(Veh_composition_number).VehCompRelFlows.GetAll()
                rel_100 = Rel_Flows[0].AttValue("RelFlow")
                rel_300 = Rel_Flows[1].AttValue("RelFlow")
                rel_630 = Rel_Flows[2].AttValue("RelFlow")
                rel_640 = Rel_Flows[3].AttValue("RelFlow")
                rel_650 = Rel_Flows[4].AttValue("RelFlow")

                print("교통량 : ", sim_volume)
                print("진입부 : ", new_gradient, "진출부 : ", new_gradient2)
                print("100rel: ", round(rel_100,4), "300rel : ", round(rel_300,4), "630rel : ", round(rel_630,4), "640rel : ", round(rel_640,4), "650rel : ", round(rel_650,4))


                print("VISSIM 시뮬레이션 실행 중...")
                Vissim.Simulation.RunContinuous()

                Vissim.Simulation.Stop()
                print("VISSIM 시뮬레이션 종료")
                print("==============================================================================================================")

file_path :  E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\ToyNetwork\2\ToyNetwork_260226.inpx
교통량 :  1162.0
진입부 :  -0.03 진출부 :  0.03
100rel:  0.7262 300rel :  0.0738 630rel :  0.1426 640rel :  0.0507 650rel :  0.0068
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 종료
교통량 :  1162.0
진입부 :  -0.05 진출부 :  0.05
100rel:  0.7262 300rel :  0.0738 630rel :  0.1426 640rel :  0.0507 650rel :  0.0068
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 종료
교통량 :  1162.0
진입부 :  -0.07 진출부 :  0.07
100rel:  0.7262 300rel :  0.0738 630rel :  0.1426 640rel :  0.0507 650rel :  0.0068
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 종료
교통량 :  1162.0
진입부 :  -0.09 진출부 :  0.09
100rel:  0.7262 300rel :  0.0738 630rel :  0.1426 640rel :  0.0507 650rel :  0.0068
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 종료
교통량 :  1154.0
진입부 :  -0.03 진출부 :  0.03
100rel:  0.6809 300rel :  0.0691 630rel :  0.1782 640rel :  0.0634 650rel :  0.0084
VISSIM 시뮬레이션 실행 중...
VISSIM 시뮬레이션 종료
교통량 :  1154.0
진입부 :  -0.05 진출부 :  0.05
100rel:  0.6809 300rel :  0.0691 630rel :  0.1782